# Databricks → Neo4j: Load

**Run on dedicated compute. Pipeline step 1 of 3 (slide 12: "The Enrichment Pipeline").**

Load Silver into Neo4j. The six Silver tables leave Unity Catalog and become a property graph in Neo4j Aura: `:Account` and `:Merchant` nodes, `TRANSFERRED_TO` relationships between accounts, and `TRANSACTED_WITH` relationships between accounts and merchants. Section 8 then adds the KYC identity layer — `:Customer` / `:Phone` / `:Address` nodes joined by `OWNS` / `HAS_PHONE` / `HAS_ADDRESS`. This is the graph that the next notebook (`04_gds_enrichment`) projects and runs algorithms against, and that `06_kyc_walkthrough` resolves shared identities over.

The same records. A different data structure. A graph database doesn't need a starting account ID to search — it takes a pattern and returns every place the pattern appears in the network. That capability is why the connection questions on slide 10 live in the graph layer and not in SQL.

In [ ]:
%pip install graphdatascience --quiet

In [ ]:
dbutils.library.restartPython()

## 0. Configuration

Run after the `%pip install` kernel restart above.

In [ ]:
CATALOG   = "graph-on-databricks"
SCHEMA    = "graph-enriched-schema"

NEO4J_URI      = dbutils.secrets.get("neo4j-graph-engineering", "uri")
NEO4J_USER     = dbutils.secrets.get("neo4j-graph-engineering", "username")
NEO4J_PASSWORD = dbutils.secrets.get("neo4j-graph-engineering", "password")

# Common Spark Connector options
NEO4J_OPTS = {
    "url":                    NEO4J_URI,
    "authentication.basic.username": NEO4J_USER,
    "authentication.basic.password": NEO4J_PASSWORD,
    "batch.size":             "10000",
}

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")

## 2. Clear Neo4j (idempotent re-runs)

Optional: wipe previous graph so the demo is repeatable.

In [ ]:
from graphdatascience import GraphDataScience

gds = GraphDataScience(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
gds.run_cypher("MATCH (n) DETACH DELETE n")
print("Neo4j cleared.")

## 3. Write Account Nodes

In [ ]:
from pyspark.sql import functions as F

accounts_df = spark.table(f"`{CATALOG}`.`{SCHEMA}`.accounts")
n_accounts = accounts_df.count()

(accounts_df
 .write
 .format("org.neo4j.spark.DataSource")
 .mode("Append")
 .options(**NEO4J_OPTS)
 .option("labels", ":Account")
 .option("node.keys", "account_id")
 .save()
)

print(f"Wrote {n_accounts} Account nodes")

## 4. Write Merchant Nodes

In [ ]:
merchants_df = spark.table(f"`{CATALOG}`.`{SCHEMA}`.merchants")
n_merchants = merchants_df.count()

(merchants_df
 .write
 .format("org.neo4j.spark.DataSource")
 .mode("Append")
 .options(**NEO4J_OPTS)
 .option("labels", ":Merchant")
 .option("node.keys", "merchant_id")
 .save()
)

print(f"Wrote {n_merchants} Merchant nodes")

## 5. Create Indexes Before Relationship Writes

Uniqueness constraints also create an index. Without them the Spark Connector
does a full node scan per relationship row. Indexes are required for fast
`account_id` and `merchant_id` lookups when writing relationships.

The same block also declares the keys for the KYC identity layer
(`Customer` / `Phone` / `Address`) and the knowledge layer
(`Policy` / `BusinessTerm` / `BusinessRule` / `DataSource`). The `Phone` and
`Address` uniqueness constraints must exist **before** the `HAS_PHONE` /
`HAS_ADDRESS` writes in section 8, so that customers who share an identifier
MERGE onto one node instead of creating duplicates. The knowledge-layer nodes
themselves are built later, in `06_kyc_walkthrough`, next to the classification
that uses them; only their keys are declared here alongside the others.

In [ ]:
gds.run_cypher("""
    CREATE CONSTRAINT account_id_unique IF NOT EXISTS
    FOR (a:Account) REQUIRE a.account_id IS UNIQUE
""")
gds.run_cypher("""
    CREATE CONSTRAINT merchant_id_unique IF NOT EXISTS
    FOR (m:Merchant) REQUIRE m.merchant_id IS UNIQUE
""")

# Identity-layer keys. phone_number_unique / address_address_unique must exist
# before the HAS_PHONE / HAS_ADDRESS writes so shared identifiers MERGE onto a
# single node.
gds.run_cypher("""
    CREATE CONSTRAINT customer_id_unique IF NOT EXISTS
    FOR (c:Customer) REQUIRE c.customer_id IS UNIQUE
""")
gds.run_cypher("""
    CREATE CONSTRAINT phone_number_unique IF NOT EXISTS
    FOR (p:Phone) REQUIRE p.number IS UNIQUE
""")
gds.run_cypher("""
    CREATE CONSTRAINT address_address_unique IF NOT EXISTS
    FOR (a:Address) REQUIRE a.address IS UNIQUE
""")

# Knowledge-layer keys. The Policy/BusinessTerm/BusinessRule/DataSource nodes
# are created in 06_kyc_walkthrough alongside the WCC classification; their
# uniqueness constraints live here with the other node keys.
gds.run_cypher("""
    CREATE CONSTRAINT policy_id_unique IF NOT EXISTS
    FOR (p:Policy) REQUIRE p.policy_id IS UNIQUE
""")
gds.run_cypher("""
    CREATE CONSTRAINT business_term_name_unique IF NOT EXISTS
    FOR (t:BusinessTerm) REQUIRE t.name IS UNIQUE
""")
gds.run_cypher("""
    CREATE CONSTRAINT business_rule_id_unique IF NOT EXISTS
    FOR (r:BusinessRule) REQUIRE r.rule_id IS UNIQUE
""")
gds.run_cypher("""
    CREATE CONSTRAINT data_source_name_unique IF NOT EXISTS
    FOR (d:DataSource) REQUIRE d.name IS UNIQUE
""")
print("Indexes ready.")

## 6. Write TRANSACTED_WITH Relationships (Account → Merchant)

In [ ]:
txn_df = spark.table(f"`{CATALOG}`.`{SCHEMA}`.transactions")
n_txns = txn_df.count()

(txn_df
 .write
 .format("org.neo4j.spark.DataSource")
 .mode("Overwrite")
 .options(**NEO4J_OPTS)
 .option("relationship", "TRANSACTED_WITH")
 .option("relationship.save.strategy", "keys")
 .option("relationship.source.labels", ":Account")
 .option("relationship.source.node.keys", "account_id:account_id")
 .option("relationship.target.labels", ":Merchant")
 .option("relationship.target.node.keys", "merchant_id:merchant_id")
 .save()
)

print(f"Wrote {n_txns} TRANSACTED_WITH relationships")

## 7. Write TRANSFERRED_TO Relationships (Account → Account)

In [ ]:
p2p_df = spark.table(f"`{CATALOG}`.`{SCHEMA}`.account_links")
n_p2p = p2p_df.count()

(p2p_df
 .write
 .format("org.neo4j.spark.DataSource")
 .mode("Overwrite")
 .options(**NEO4J_OPTS)
 .option("relationship", "TRANSFERRED_TO")
 .option("relationship.save.strategy", "keys")
 .option("relationship.source.labels", ":Account")
 .option("relationship.source.node.keys", "src_account_id:account_id")
 .option("relationship.target.labels", ":Account")
 .option("relationship.target.node.keys", "dst_account_id:account_id")
 .save()
)

print(f"Wrote {n_p2p} TRANSFERRED_TO relationships")

## 8. Add the KYC Identity Layer

The transfer graph above answers "who moves money to whom." The KYC story needs
a second structure: "who *is* whom." We load the `customers` table as `:Customer`
nodes, then turn each customer's phone and address into their own `:Phone` and
`:Address` nodes.

Phone and address are deliberately **not** stored as customer properties. Because
the `HAS_PHONE` / `HAS_ADDRESS` writes MERGE onto a single node per value
(`relationship.target.save.mode = Overwrite` against the uniqueness constraints
from section 5), two customers who share a phone number both point at the *same*
`:Phone` node. Sharing an identifier becomes graph structure you can traverse,
not a value you have to self-join to discover. `06_kyc_walkthrough` runs Weakly
Connected Components over exactly this layer to resolve shared-identity rings.

In [ ]:
customers_df = spark.table(f"`{CATALOG}`.`{SCHEMA}`.customers")
n_customers = customers_df.count()

# Write Customer nodes. phone and address are left off the node on purpose —
# they become :Phone / :Address nodes via the relationship writes below.
(customers_df
 .selectExpr("customer_id", "customer_name AS name", "email")
 .write
 .format("org.neo4j.spark.DataSource")
 .mode("Append")
 .options(**NEO4J_OPTS)
 .option("labels", ":Customer")
 .option("node.keys", "customer_id")
 .save()
)

print(f"Wrote {n_customers} Customer nodes")

In [ ]:
# OWNS: Customer → Account (one owner per account)
(customers_df
 .select("customer_id", "account_id")
 .write
 .format("org.neo4j.spark.DataSource")
 .mode("Overwrite")
 .options(**NEO4J_OPTS)
 .option("relationship", "OWNS")
 .option("relationship.save.strategy", "keys")
 .option("relationship.source.labels", ":Customer")
 .option("relationship.source.node.keys", "customer_id:customer_id")
 .option("relationship.target.labels", ":Account")
 .option("relationship.target.node.keys", "account_id:account_id")
 .save()
)

print(f"Wrote {n_customers} OWNS relationships (Customer → Account)")

In [ ]:
# HAS_PHONE: Customer → Phone. target.save.mode Overwrite MERGEs :Phone on
# number, so customers sharing a phone converge on one node. coalesce(1)
# serializes the write: two partitions MERGEing the same shared value
# concurrently would race on node creation and one would fail the uniqueness
# constraint.
(customers_df
 .select("customer_id", "phone")
 .coalesce(1)
 .write
 .format("org.neo4j.spark.DataSource")
 .mode("Overwrite")
 .options(**NEO4J_OPTS)
 .option("relationship", "HAS_PHONE")
 .option("relationship.save.strategy", "keys")
 .option("relationship.source.labels", ":Customer")
 .option("relationship.source.node.keys", "customer_id:customer_id")
 .option("relationship.target.labels", ":Phone")
 .option("relationship.target.node.keys", "phone:number")
 .option("relationship.target.save.mode", "Overwrite")
 .save()
)

print("Wrote HAS_PHONE relationships (Customer → Phone)")

In [ ]:
# HAS_ADDRESS: Customer → Address. Same shared-target MERGE and coalesce(1)
# serialization as HAS_PHONE.
(customers_df
 .select("customer_id", "address")
 .coalesce(1)
 .write
 .format("org.neo4j.spark.DataSource")
 .mode("Overwrite")
 .options(**NEO4J_OPTS)
 .option("relationship", "HAS_ADDRESS")
 .option("relationship.save.strategy", "keys")
 .option("relationship.source.labels", ":Customer")
 .option("relationship.source.node.keys", "customer_id:customer_id")
 .option("relationship.target.labels", ":Address")
 .option("relationship.target.node.keys", "address:address")
 .option("relationship.target.save.mode", "Overwrite")
 .save()
)

print("Wrote HAS_ADDRESS relationships (Customer → Address)")

## 9. Verify Graph in Neo4j

In [ ]:
result = gds.run_cypher("""
    CALL apoc.meta.stats() YIELD nodeCount, relCount, labels, relTypes
    RETURN nodeCount, relCount, labels, relTypes
""")
display(result)

### Quick Counts

In [ ]:
counts = gds.run_cypher("""
    MATCH (a:Account) WITH count(a) AS accounts
    MATCH (m:Merchant) WITH accounts, count(m) AS merchants
    MATCH (c:Customer) WITH accounts, merchants, count(c) AS customers
    MATCH (ph:Phone) WITH accounts, merchants, customers, count(ph) AS phones
    MATCH (ad:Address)
    WITH accounts, merchants, customers, phones, count(ad) AS addresses
    MATCH ()-[t:TRANSACTED_WITH]->()
    WITH accounts, merchants, customers, phones, addresses, count(t) AS txns
    MATCH ()-[p:TRANSFERRED_TO]->()
    WITH accounts, merchants, customers, phones, addresses, txns, count(p) AS p2p
    RETURN accounts, merchants, customers, phones, addresses, txns, p2p
""")
print(counts.to_string(index=False))

### Sample: One Account's Neighbourhood

In [ ]:
sample = gds.run_cypher("""
    MATCH (a:Account)-[r]->(target)
    WITH a, type(r) AS rel_type, labels(target)[0] AS target_type, count(*) AS cnt
    RETURN a.account_id AS account, rel_type, target_type, cnt
    ORDER BY cnt DESC
    LIMIT 10
""")
display(spark.createDataFrame(sample))

**What we just did:** a handful of write operations pushed our entire Delta Lake dataset into Neo4j
as a typed property graph — the transfer network (`:Account` / `:Merchant`) plus the KYC identity
layer (`:Customer` / `:Phone` / `:Address`) — with no ETL pipeline, no CSV exports, and no Cypher
`LOAD CSV`. Customers who share a phone or address already point at the same identifier node.

**Next →** `04_gds_enrichment`: run graph algorithms to extract fraud-signal features. Then `06_kyc_walkthrough` resolves the shared-identity rings sitting in the identity layer we just loaded.